# 05 · Modelo predictivo de resultados (16.4, opcional)

Regresión logística multinomial (victoria, empate o derrota del local) con variables conocidas antes del partido: diferencia de Elo con localía, forma de los últimos 5 y días de descanso. Validación temporal (`TimeSeriesSplit`, sin fuga de datos) contra el Elo solo, con el Elo ya calibrado en el notebook 02 (K = 30, V = 50): comparar contra un Elo mal calibrado inflaría el modelo. **Se publica únicamente si le gana al Elo.**

In [1]:
import sys, statistics, math
sys.path.insert(0, '.')
from datos import api, league_games, result, log_loss, elo_probabilities, m, RESULTS
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
import json
import numpy as np
from datetime import date
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
K, V = 30, 50  # calibrados en 02_calibracion
juegos = league_games()
pre = m.elo_run(juegos, K, V)
ultimos, forma = {}, {}
X, y = [], []
for g, (rh, ra) in zip(juegos, pre):
    d = date.fromisoformat(g.date)
    descanso = [min((d - ultimos[t]).days, 14) if t in ultimos else 14 for t in (g.home, g.away)]
    f = [sum(forma.get(t, [])[-5:]) for t in (g.home, g.away)]
    X.append([rh - ra + V, f[0] - f[1], descanso[0] - descanso[1]])
    y.append(RESULTS.index(result(g)))
    ph = 3 if g.home_goals > g.away_goals else 1 if g.home_goals == g.away_goals else 0
    pa = 3 if ph == 0 else 1 if ph == 1 else 0
    for t, pt in ((g.home, ph), (g.away, pa)):
        ultimos[t] = d
        forma.setdefault(t, []).append(pt)
X, y = np.array(X[60:], float), np.array(y[60:])
print(f"{len(y)} partidos con variables (los primeros 60 acomodan el Elo y la forma)")

404 partidos con variables (los primeros 60 acomodan el Elo y la forma)


In [3]:
def ll(P, yy):
    return float(np.mean(-np.log(np.clip(P[np.arange(len(yy)), yy], 1e-12, 1))))
def br(P, yy):
    return float(np.mean(np.sum((P - np.eye(3)[yy]) ** 2, axis=1)))
filas = []
for tr, te in TimeSeriesSplit(n_splits=5).split(X):
    pm = LogisticRegression(max_iter=1000).fit(X[tr], y[tr]).predict_proba(X[te])
    empates = float(np.mean(y[tr] == 1))
    pe = np.array([m.outcome_probabilities(1 / (1 + 10 ** (-x[0] / 400)), empates) for x in X[te]])
    filas.append((ll(pm, y[te]), ll(pe, y[te]), br(pm, y[te]), br(pe, y[te])))
    print(f"Pliegue de {len(te)} partidos · log-loss modelo {filas[-1][0]:.4f} vs Elo {filas[-1][1]:.4f} · Brier {filas[-1][2]:.4f} vs {filas[-1][3]:.4f}")
media = np.mean(filas, axis=0)
gana = sum(a < b for a, b, _, _ in filas)
publicar = bool(media[0] < media[1] and gana >= 4)
print(f"Media · log-loss modelo {media[0]:.4f} vs Elo {media[1]:.4f} · gana en {gana} de 5 pliegues")
print('Decisión:', 'se publica (le gana al Elo)' if publicar else 'NO se publica: no le gana al Elo de forma consistente')

Pliegue de 67 partidos · log-loss modelo 1.1969 vs Elo 1.1224 · Brier 0.7324 vs 0.6767
Pliegue de 67 partidos · log-loss modelo 1.0802 vs Elo 1.0784 · Brier 0.6537 vs 0.6520


Pliegue de 67 partidos · log-loss modelo 1.0456 vs Elo 1.0565 · Brier 0.6266 vs 0.6341
Pliegue de 67 partidos · log-loss modelo 1.0752 vs Elo 1.0986 · Brier 0.6503 vs 0.6682


Pliegue de 67 partidos · log-loss modelo 1.0204 vs Elo 0.9803 · Brier 0.6108 vs 0.5835
Media · log-loss modelo 1.0836 vs Elo 1.0672 · gana en 2 de 5 pliegues
Decisión: NO se publica: no le gana al Elo de forma consistente


In [4]:
final = LogisticRegression(max_iter=1000).fit(X, y)
ficha = {'variables': ['diferencia de Elo con localía', 'forma local − visitante (puntos de los últimos 5)', 'descanso local − visitante (días, tope 14)'],
         'clases': list(RESULTS), 'coeficientes': final.coef_.round(4).tolist(), 'intercepto': final.intercept_.round(4).tolist(),
         'n': int(len(y)), 'log_loss_modelo': round(float(media[0]), 4), 'log_loss_elo': round(float(media[1]), 4), 'publicado': publicar}
print(json.dumps(ficha, ensure_ascii=False, indent=1))

{
 "variables": [
  "diferencia de Elo con localía",
  "forma local − visitante (puntos de los últimos 5)",
  "descanso local − visitante (días, tope 14)"
 ],
 "clases": [
  "G",
  "E",
  "P"
 ],
 "coeficientes": [
  [
   0.0054,
   -0.0068,
   0.0212
  ],
  [
   -0.0013,
   -0.0157,
   -0.0729
  ],
  [
   -0.0041,
   0.0224,
   0.0517
  ]
 ],
 "intercepto": [
  -0.0193,
  -0.0633,
  0.0826
 ],
 "n": 404,
 "log_loss_modelo": 1.0836,
 "log_loss_elo": 1.0672,
 "publicado": false
}

**Lectura.** Los coeficientes se guardan como JSON (no `pickle`) para poder explicarlos; la ficha del modelo está en `docs/model_cards/resultado_partido.md`.